# Region-level OCR, Tesseract part (third review C5), Colab CPU

The Kaggle CPU run (D42-D43) gave usable EasyOCR region text, but its conda-forge Tesseract 5.3.4 reads
differently from the Colab Tesseract of the main run. This notebook re-reads only the Tesseract part on
Colab, keeps the EasyOCR region text of the Kaggle run, and first checks that Colab's Tesseract still
reproduces the main run's whole-page OCR (it stops if fewer than 38 of 40 pages match).

**Runtime:** CPU (Runtime > Change runtime type > CPU). About 30-45 min. Keep the tab open.

**Needs on Google Drive, in `My Drive/bengali-postocr-results/`:** `ocr_eval40.jsonl` and
`ocr_region40.jsonl` (from the Kaggle `region.zip`). `My Drive/Dataset/REID2019.zip` as before.
Output: `ocr_region40_colab.jsonl` and `ocr_region40_colab_verify.json` in the same folder.


In [ ]:
import os, json
REPO_DIR = '/content/bengali-postocr-sllm'
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/Ariffurrhmn/bengali-postocr-sllm.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
!cd {REPO_DIR} && git log --oneline -1
!apt-get -qq update && apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q pytesseract easyocr==1.7.2 jiwer
!mkdir -p {REPO_DIR}/.tessdata
for lang in ('ben', 'eng', 'osd'):
    !curl -sL -o {REPO_DIR}/.tessdata/{lang}.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/{lang}.traineddata
!tesseract --version | head -1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/My Drive/bengali-postocr-results'
DATASET_DIR = '/content/Competition_dataset_ImagesPAGEXML'
!unzip -q -o "/content/drive/My Drive/Dataset/REID2019.zip" -d /content
for f in ('ocr_eval40.jsonl', 'ocr_region40.jsonl'):
    assert os.path.exists(f'{RESULTS_DIR}/{f}'), f'copy {f} to My Drive/bengali-postocr-results/'


In [ ]:
os.environ['TESSDATA_PREFIX'] = f'{REPO_DIR}/.tessdata'
os.environ['TESSERACT_CMD'] = 'tesseract'
!cd {REPO_DIR}/ocr && python run_region_ocr.py --dataset-dir "{DATASET_DIR}" \
    --stored "{RESULTS_DIR}/ocr_eval40.jsonl" --verify --tesseract-only \
    --keep-easyocr "{RESULTS_DIR}/ocr_region40.jsonl" --out "{RESULTS_DIR}/ocr_region40_colab.jsonl"


In [ ]:
v = json.load(open(f'{RESULTS_DIR}/ocr_region40_colab_verify.json'))
print('versions', v['versions'], '| Tesseract identical to the main run on', v['tesseract_identical'], 'of', v['pages'])
rows = [json.loads(l) for l in open(f'{RESULTS_DIR}/ocr_region40_colab.jsonl', encoding='utf-8') if l.strip()]
print(len(rows), 'pages,', sum(len(r['regions']) for r in rows), 'regions')
